# HW 08: Vision & Sensory Delay
**Computational Sensorimotor Control — Week 8**

**Problems:**
1. Torque decomposition — why delay causes y-deviation on a horizontal reach
2. Why not PD? — the velocity noise problem on the arc
3. Gaze strategy simulation — reproducing Figure 8 with proper saccade costs
4. Short answer — connecting simulation results to Elliott et al.'s model

**Submit:** This notebook (with output) + written answers in HW08_Assignment.docx


In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import ceil
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

from smc import Arm, Q_REF, mass_matrix, coriolis, joint_accelerations, Sensor

arm = Arm()
ik  = arm.inverse_kinematics
fk  = arm.forward_kinematics
Jfn = arm.jacobian
start_hand = np.array(fk(Q_REF))
dt = 0.001
k_noise = 0.15

def minjerk_basis(t, T):
    tau = np.clip(t / T, 0, 1)
    return 10*tau**3 - 15*tau**4 + 6*tau**5

def minjerk_reach(target, T, dt=0.001):
    t = np.arange(0, T + dt/2, dt)
    s = minjerk_basis(t, T)
    return t, start_hand[None,:] + s[:,None] * (target - start_hand)[None,:]

def minjerk_arc(R, T, dt=0.001):
    center = start_hand + np.array([R, 0])
    t = np.arange(0, T + dt/2, dt)
    s = minjerk_basis(t, T)
    th = np.pi * (1 - s)
    return t, np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def id_torques(t_arr, pos):
    n = len(t_arr)
    q = np.array([ik(pos[i,0], pos[i,1]) for i in range(n)])
    qd = np.gradient(q, dt, axis=0); qdd = np.gradient(qd, dt, axis=0)
    tau = np.zeros((n, 2))
    for i in range(n): tau[i] = mass_matrix(q[i]) @ qdd[i] + coriolis(q[i], qd[i])
    return q, qd, qdd, tau

def add_sdn(tau, k, rng):
    return tau + k * np.abs(tau) * rng.standard_normal(2)

# ── Controllers from Lab 08 ──
class _Sensor:
    def __init__(s,delay,sigma): s.bl=max(1,ceil(delay/dt)); s.sigma=sigma; s.buf=[]
    def reset(s): s.buf=[]
    def sense(s,x,rng):
        s.buf.append(x.copy()); idx=max(0,len(s.buf)-1-s.bl)
        return s.buf[idx]+rng.normal(0,s.sigma,len(x))

def sim_delay_pd(q0, tau_ff, pos_des, vel_des, Kp, Kd, delay_ms, sigma, rng,
                sample_interval_ms=10):
    """FF + PD with sensor sampled at sample_interval_ms. Returns hand, tau_corr."""
    sen = _Sensor(delay_ms/1000, sigma); sen.reset()
    n = len(tau_ff); si = int(sample_interval_ms / (dt*1000))
    q=np.zeros((n,2)); qd=np.zeros((n,2)); hand=np.zeros((n,2))
    tau_corr=np.zeros((n,2))
    q[0]=q0.copy(); hand[0]=fk(q0); prev_y=hand[0].copy()
    last_tc = np.zeros(2)
    for i in range(n-1):
        if i % si == 0:  # sample at interval
            y = sen.sense(hand[i], rng)
            yd = (y - prev_y) / (si * dt) if i > 0 else np.zeros(2)
            prev_y = y.copy()
            ep = pos_des[min(i,len(pos_des)-1)] - y
            ev = vel_des[min(i,len(vel_des)-1)] - yd
            last_tc = Jfn(q[i]).T @ (Kp*ep + Kd*ev)
        else:
            sen.sense(hand[i], rng)  # keep buffer current
        tau_corr[i] = last_tc
        tt = tau_ff[i] + last_tc
        qdd = joint_accelerations(q[i], qd[i], tt)
        q[i+1]=q[i]+qd[i]*dt+0.5*qdd*dt**2
        qd[i+1]=qd[i]+qdd*dt; hand[i+1]=fk(q[i+1])
    return hand, tau_corr

def sim_delay_p(q0, tau_ff, pos_des, Kp, delay_s, sigma, rng):
    n=len(tau_ff); q=np.zeros((n,2)); qd=np.zeros((n,2)); hand=np.zeros((n,2))
    q[0]=q0.copy(); hand[0]=fk(q0); buf=[]; bl=int(delay_s/dt)
    for i in range(n-1):
        buf.append(hand[i].copy()); tc=np.zeros(2)
        if Kp>0 and len(buf)>bl:
            y=buf[-bl-1]+rng.normal(0,sigma,2)
            tc=Jfn(q[i]).T@(Kp*(pos_des[max(0,i-bl)]-y))
        tt=add_sdn(tau_ff[i]+tc, k_noise, rng)
        qdd=joint_accelerations(q[i],qd[i],tt)
        q[i+1]=q[i]+qd[i]*dt+0.5*qdd*dt**2
        qd[i+1]=qd[i]+qdd*dt; hand[i+1]=fk(q[i+1])
    return hand

NAVY='#1B2A4A'; TEAL='#2E86AB'; RED='#E74C3C'
GREEN='#27AE60'; GRAY='#7F8C8D'; ORANGE='#E67E22'; PURPLE='#8E44AD'
print('Setup complete.')


---
## Problem 1: Torque Decomposition (builds on Lab Part 2)

Run a single trial of the PD controller at Δ=0, 100, 200 ms on the same 10 cm reach
from Lab Part 2 (Kp=5, Kd=1.5, σ=0.3mm, T=600ms). The `sim_delay_pd` in the setup
has been modified to return `tau_corr` alongside `hand`.

To keep the correction signal readable, the controller samples the sensor at 10 ms intervals
(consistent with Figure 4 in the lecture) rather than every 1 ms timestep. Between samples,
the correction torque is held constant. This also makes the velocity estimate realistic:
σ_vel = 0.3mm / 10ms = 0.01 m/s, negligible compared to peak hand velocity (~0.3 m/s).

**Your tasks:**
1. For each delay, plot τ_ff, τ_corr, and τ_total for shoulder and elbow — 6 subplots (3 delays × 2 joints).
2. At the midpoint of the movement (t ≈ 300 ms), print J^T for the current configuration. Which element maps an x-position error to elbow torque?
3. In your written answer (HW08_Assignment.docx), explain in 3–4 sentences why a delayed x-error produces y-deviation, referencing the off-diagonal element of J^T.


In [ ]:
# ── Problem 1: Torque decomposition ──
tgt1 = start_hand + np.array([0.10, 0]); T1 = 0.6
t1, pos1 = minjerk_reach(tgt1, T1)
vel1 = np.gradient(pos1, dt, axis=0)
q1, _, _, tau1 = id_torques(t1, pos1)

# YOUR CODE HERE
# Run sim_delay_pd at three delays (0, 100, 200 ms)
# Plot τ_ff, τ_corr, τ_total for shoulder and elbow
# Print J^T at t=300ms


---
## Problem 2: Why Not PD on the Arc?

In Lab Part 3, we used P-only correction (Kp=8) because the D-term fails with noisy sensors.
Now demonstrate this failure.

**Your tasks:**
1. Compute the velocity noise SD analytically: σ_vel = σ_pos / Δt_sample. Fill in the table below for σ=1mm (central) and σ=5mm (peripheral) at Δt_sample = 1ms and 33ms.
2. Add a Kd term to `sim_delay_p` (provided as `sim_delay_pd_arc` below). Run 200 trials on the 6cm arc (T=800ms) with Kd=0 (P-only) and Kd=0.05 for both channels.
3. Produce a table of endpoint SDs comparing OL, P-only, and PD for both channels.
4. In your written answer, explain in 3–4 sentences why the D-term fails and what would fix it.


In [ ]:
# ── Analytical velocity noise ──
print('Velocity noise SD = σ_pos / Δt_sample')
print(f'{"":>15} {"Δt=1ms":>10} {"Δt=33ms":>10} {"Peak vel":>10}')
# YOUR CODE HERE: fill in the four values
# print(f'{"Central σ=1mm":>15} {???:>10.3f} {???:>10.3f} {"~0.3 m/s":>10}')
# print(f'{"Periph σ=5mm":>15} {???:>10.3f} {???:>10.3f} {"~0.3 m/s":>10}')


In [ ]:
# ── sim_delay_pd_arc: P or PD on the arc ──
def sim_delay_pd_arc(q0, tau_ff, pos_des, Kp, Kd, delay_s, sigma, rng):
    """Like sim_delay_p but with optional Kd (estimated velocity via differentiation)."""
    n = len(tau_ff)
    q=np.zeros((n,2)); qd=np.zeros((n,2)); hand=np.zeros((n,2))
    q[0]=q0.copy(); hand[0]=fk(q0)
    buf=[]; bl=int(delay_s/dt); prev_y=hand[0].copy()
    for i in range(n-1):
        buf.append(hand[i].copy()); tc=np.zeros(2)
        if Kp>0 and len(buf)>bl:
            y=buf[-bl-1]+rng.normal(0,sigma,2)
            ep=pos_des[max(0,i-bl)]-y
            yd=(y-prev_y)/dt; prev_y=y.copy()
            tc=Jfn(q[i]).T@(Kp*ep - Kd*yd)
        tt=add_sdn(tau_ff[i]+tc, k_noise, rng)
        qdd=joint_accelerations(q[i],qd[i],tt)
        q[i+1]=q[i]+qd[i]*dt+0.5*qdd*dt**2
        qd[i+1]=qd[i]+qdd*dt; hand[i+1]=fk(q[i+1])
    return hand
print('sim_delay_pd_arc defined.')


In [ ]:
# ── Problem 2: PD failure on the arc ──
R_ARC = 0.06; T_ARC = 0.8; Kp = 8; N = 200
t_a, pos_a = minjerk_arc(R_ARC, T_ARC)
q_a, _, _, tau_a = id_torques(t_a, pos_a)
tgt_a = pos_a[-1]

# YOUR CODE HERE
# Run sim_delay_pd_arc with Kd=0 and Kd=0.05 for both channels
# Print a table: OL, P-only (CV), P-only (PV), PD (CV, Kd=0.05), PD (PV, Kd=0.05)


---
## Problem 3: Gaze Strategy Simulation (Lecture §5 — Figure 8)

Implement three gaze strategies on a 10 cm center-out reach (T=600ms, Kp=8, P-only):

- **Fixate target:** gaze on target, hand in periphery (Δ=80ms, σ=5mm) for the entire movement
- **Track hand:** gaze on hand (Δ=150ms, σ=1mm), but target position known with peripheral noise (σ=5mm)
- **Saccade switch:** peripheral correction 0–300ms → 40ms saccadic suppression (blind) → 100ms fixation stabilization → central correction from 440ms onward. First central measurement arrives at 440+150=590ms.

**Your tasks:**
1. Implement `sim_gaze` below — the fixate-target and track-hand branches are provided. Fill in the saccade switch.
2. Run 200 trials for each strategy. Produce a 4-panel figure (3 scatter + 1 bar chart).
3. In your written answer, explain why fixate-target wins and why the saccade switch can't beat it.


In [ ]:
def sim_gaze(q0, tau_ff, pos_des, tgt, Kp, rng, strategy='fixate_target',
             saccade_time_ms=300, saccade_dur_ms=40, fixation_stab_ms=100):
    n = len(tau_ff)
    q=np.zeros((n,2)); qd=np.zeros((n,2)); hand=np.zeros((n,2))
    q[0]=q0.copy(); hand[0]=fk(q0)
    buf=[]; bl_p=int(0.080/dt); bl_c=int(0.150/dt)
    sig_f=0.001; sig_p=0.005
    s_start=int(saccade_time_ms/dt)
    fix_ready=s_start + int(saccade_dur_ms/dt) + int(fixation_stab_ms/dt)

    for i in range(n-1):
        buf.append(hand[i].copy()); tc=np.zeros(2)

        if strategy == 'fixate_target':
            if Kp>0 and len(buf)>bl_p:
                y=buf[-bl_p-1]+rng.normal(0,sig_p,2)
                tc=Jfn(q[i]).T@(Kp*(pos_des[max(0,i-bl_p)]-y))

        elif strategy == 'track_hand':
            if Kp>0 and len(buf)>bl_c:
                y=buf[-bl_c-1]+rng.normal(0,sig_f,2)
                des_noisy=pos_des[max(0,i-bl_c)]+rng.normal(0,sig_p,2)
                tc=Jfn(q[i]).T@(Kp*(des_noisy-y))

        elif strategy == 'saccade_switch':
            # YOUR CODE HERE
            # Phase 1 (i < s_start): peripheral correction like fixate_target
            # Phase 2 (s_start <= i < fix_ready): blind — tc stays zeros
            # Phase 3 (i >= fix_ready): central correction like central channel
            pass

        tt = add_sdn(tau_ff[i]+tc, k_noise, rng)
        qdd=joint_accelerations(q[i],qd[i],tt)
        q[i+1]=q[i]+qd[i]*dt+0.5*qdd*dt**2
        qd[i+1]=qd[i]+qdd*dt; hand[i+1]=fk(q[i+1])
    return hand

print('sim_gaze defined.')


In [ ]:
# ── Problem 3: Run and plot ──
tgt3 = start_hand + np.array([0.10, 0]); T3 = 0.6; Kp = 8; N3 = 200
t3, pos3 = minjerk_reach(tgt3, T3)
q3, _, _, tau3 = id_torques(t3, pos3)

# YOUR CODE HERE
# Run sim_gaze for each strategy, collect endpoints, produce 4-panel figure


---
## Problem 4: Short Answer

Answer the following in HW08_Assignment.docx (2–4 sentences each):

**(a)** Why does the nervous system typically fixate the target rather than the hand during reaching?
Connect your answer to Elliott et al.'s two control processes (impulse control and limb-target control).

**(b)** List the three things missing from our Week 8 controller (the three gaps identified in §7
of the lecture). For each, state which week addresses it.

**(c)** How would a forward model (internal model that predicts sensory consequences of motor commands)
solve the PD failure you demonstrated in Problem 2? Be specific about what the forward model
provides that raw differentiation does not.


---
## Submission

Submit:
1. This notebook with all cells executed (Problems 1–3)
2. HW08_Assignment.docx with written answers for Problems 1.3, 2.4, 3.3, and 4(a–c)
